In [1]:
from Objects.Transformations import *
from Objects.WSBM import *
from Objects.TWSBMInstance import *

from Computation.Computation import *
from Computation.ExtraMetrics import *

from Plotting.Plotting import *
from Plotting.ArtisticPlotting import *
import cloudpickle

emb_mode = 'sqrt-scaled'
p22 = 'fixed'
n_batch = 5
eps = True

def sample_in_rectangle(x_a=0, x_b=1, y_a=0, y_b=1, props=None, traversal='row-major', remove_black=False):
	props = np.asarray(props if props is not None else [0.1, 0.5, 0.9])
	N = props.size
	xs = x_a + props * (x_b - x_a)
	ys = y_a + props * (y_b - y_a)
	xx, yy = np.meshgrid(xs, ys, indexing='xy')
	ii, jj = np.meshgrid(np.arange(N), np.arange(N), indexing='xy')
	order = 'C' if traversal == 'row-major' else 'F'
	flat_x = xx.ravel(order=order)
	flat_y = yy.ravel(order=order)
	flat_i = ii.ravel(order=order)
	flat_j = jj.ravel(order=order)
	if remove_black and N >= 3:
		mask = ((flat_i + flat_j) % 2 == 0)
		flat_x = flat_x[mask]
		flat_y = flat_y[mask]
	return list(zip(flat_x, flat_y))

def plot_embeddings_grid(model, transforms, subfolder, 
						 rhos = RHOS, pis = PIS, model_params = sample_in_rectangle(), q_outliers = 0,
						 mode = 'Truth', ellipse = True, show_stats = True, seed = 0):
	emb_mode, p22 = EMB_MODES[0], P22S[0]
	metrics = {}
	for rho, pi in product(rhos, pis):
		metrics[(rho, pi)] = {}
		for p11, p12 in model_params:
			m = model(rho, pi, (p11, p12), p22 = p22)
			A, Z = m(42 + seed)
			metrics[(rho, pi)][m] = {}
			for t in transforms:
				#print(f"Simulating for rho={rho}, pi={pi}, model={model.name}, transformation={t.id}")
				metrics[(rho, pi)][m][t] = TWSBMInstance(model = m, transformation = t, A = t(A), Z = Z, emb_mode = emb_mode)
	plotter = Plotter(folder_path="", eps = eps)
	for rho, pi in product(rhos, pis):
		plotter.plot_embedding(rho, pi, metrics[(rho, pi)], subfolder = subfolder, q_outliers = q_outliers, 
						 mode = mode, ellipse = ellipse, show_stats = show_stats)
		
with open('Computation/metrics_g.cpkl', 'rb') as f:
	metrics_g = cloudpickle.load(f)
with open('Computation/best_rand_avg.cpkl', 'rb') as f:
	best_rand_avg = cloudpickle.load(f)
plotter = Plotter(folder_path="", eps = eps)

In [2]:
transforms = TRANSFORMS_QTL.copy() + [LogTransform()]
subfolder = 'C_true_Regret'


model = lognormWSBM
plot_embeddings_grid(model, transforms, f'{subfolder}/{model.name}', 
					 model_params = sample_in_rectangle(0, 1, 0.3, 0.4)[3:4], 
					 rhos = [0.1], pis = [0.1], q_outliers = 0.01)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


In [3]:
transforms = TRANSFORMS_QTL.copy() + [LogTransform()]
subfolder = 'C_embed_Regret'

model = betaWSBM
plot_embeddings_grid(model, transforms, f'{subfolder}/{model.name}', 
					 model_params = sample_in_rectangle(0.5, 0.6, 0.25, 0.4)[6:7], 
					 rhos = [0.1], pis = [0.1], q_outliers = 0.01, seed = 1)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


In [3]:
transforms = [IdentityTransform(), LogTransform(), QuantileTransform(0.25), PowerTransform(2)]
subfolder = 'Example'

model = betaWSBM
plot_embeddings_grid(model, transforms, f'{subfolder}/{model.name}', 
					 model_params = [(0.1, 0.9)], 
					 rhos = [0.1], pis = [0.25], q_outliers = 0.01,
                     ellipse = False, show_stats = False)
plot_embeddings_grid(model, transforms, f'{subfolder}/{model.name}', 
					 model_params = [(0.1, 0.9)], 
					 rhos = [0.1], pis = [0.25], q_outliers = 0.01,
                     ellipse = False, mode = 'Prediction', show_stats = False)

KeyboardInterrupt: 

In [5]:
for C in GATED_CHERNOFFS_ID:
	plotter.plot_rand_by_sigmoid_params_model_wise(C, best_rand_avg)
	plotter.plot_rand_by_sigmoid_params(C, best_rand_avg)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


In [4]:
emb_mode, p22 = EMB_MODES[0], P22S[0]
plotter = Plotter(folder_path="", eps = eps)

for model in MODELS:
	print(f"Simulating for varying_param = PowerTransform")
	plotter.plot_metrics_for_varying_param(model = model,
								   t = PowerTransform(1),
								   varying_param = PowerTransform,
								   varying_param_bounds = (0.5, 2))
	print(f"Simulating for varying_param = QuantileTransform")
	plotter.plot_metrics_for_varying_param(model = model,
								   t = QuantileTransform(0.5),
								   varying_param = QuantileTransform,
								   varying_param_bounds = (0, 0.99))

Simulating for varying_param = PowerTransform


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Simulating for varying_param = QuantileTransform


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Simulating for varying_param = PowerTransform


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Simulating for varying_param = QuantileTransform


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


In [5]:
for rho, pi, model in [(0.1, 0.1, betaWSBM), (0.1, 0.1, lognormWSBM)]:
	m = metrics_g[(rho, pi, model)]
	plotter.plot_best_transform_heatmaps(rho, pi, model, m, gated = False)
	plotter.plot_best_transform_heatmaps(rho, pi, model, m, gated = True)
	plotter.plot_best_transform_scatter_rand_vs_regretratio(rho, pi, model, m)
	
rho=0.1
pi=0.5
model=betaWSBM
plotter.plot_best_transform_scatter_rand_vs_regretratio(rho=rho, pi=pi, model=model, metrics=metrics_g[(rho, pi, model)])
	
for model in MODELS:
	plotter.plot_best_transform_scatter_rand_vs_regretratio(None, None, model, metrics_g[model])

c:\Users\Nicol\Documents\EPFL\MA6\Project\Code\Plotting\Plotting.py:541: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
c:\Users\Nicol\Documents\EPFL\MA6\Project\Code\Plotting\Plotting.py:541: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
c:\Users\Nicol\Documents\EPFL\MA6\Project\Code\Plotting\Plotting.py:541: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
The PostScript backend does not support transparency